# Cleaning: build clean_sales from raw_transactions

Runs sql/02_clean_sales.sql (rules documented there).

In [1]:
import duckdb
con = duckdb.connect("../data/retail.duckdb")

sql = open("../sql/02_clean_sales.sql").read()
con.execute(sql)

## Rows removed
64,016 rows (6.0%) were removed from raw_transactions.

In [2]:
con.sql("""WITH COUNTS AS (
            SELECT
                (SELECT COUNT(*) FROM raw_transactions) AS raw_rows,
                (SELECT COUNT(*) FROM clean_sales) AS clean_rows
)
        SELECT raw_rows,
            clean_rows,
            raw_rows - clean_rows AS removed_rows,
            ROUND(100.0 * (raw_rows - clean_rows) / raw_rows, 2) AS pct_removed
        FROM COUNTS""").show()

┌──────────┬────────────┬──────────────┬─────────────┐
│ raw_rows │ clean_rows │ removed_rows │ pct_removed │
│  int64   │   int64    │    int64     │   double    │
├──────────┼────────────┼──────────────┼─────────────┤
│  1067371 │    1003355 │        64016 │         6.0 │
└──────────┴────────────┴──────────────┴─────────────┘



## Verification checks
Each check should return 0 rows if the cleaning rules worked.

In [3]:
con.sql("""SELECT *
        FROM clean_sales
        WHERE quantity <= 0
        OR price <= 0""")

┌─────────┬────────────┬─────────────┬──────────┬──────────────┬────────┬─────────────┬─────────┬─────────┐
│ invoice │ stock_code │ description │ quantity │ invoice_date │ price  │ customer_id │ country │ revenue │
│ varchar │  varchar   │   varchar   │  int64   │  timestamp   │ double │   varchar   │ varchar │ double  │
└─────────┴────────────┴─────────────┴──────────┴──────────────┴────────┴─────────────┴─────────┴─────────┘
                                                  0 rows                                                 

In [4]:
con.sql("""SELECT *
        FROM clean_sales
        WHERE invoice LIKE 'A%' 
        OR invoice LIKE 'C%'""")

┌─────────┬────────────┬─────────────┬──────────┬──────────────┬────────┬─────────────┬─────────┬─────────┐
│ invoice │ stock_code │ description │ quantity │ invoice_date │ price  │ customer_id │ country │ revenue │
│ varchar │  varchar   │   varchar   │  int64   │  timestamp   │ double │   varchar   │ varchar │ double  │
└─────────┴────────────┴─────────────┴──────────┴──────────────┴────────┴─────────────┴─────────┴─────────┘
                                                  0 rows                                                 

In [5]:
con.sql("""SELECT *
        FROM clean_sales
        WHERE stock_code IN ('POST', 'DOT', 'AMAZONFEE', 'M', 'D', 'S', 'BANK CHARGES', 'B', 'CRUK', 'm', 'GIFT', 'C2', 'C3')
        OR stock_code LIKE 'gift_%'
        OR stock_code LIKE 'TEST%' 
        OR stock_code LIKE 'ADJUST%'""")

┌─────────┬────────────┬─────────────┬──────────┬──────────────┬────────┬─────────────┬─────────┬─────────┐
│ invoice │ stock_code │ description │ quantity │ invoice_date │ price  │ customer_id │ country │ revenue │
│ varchar │  varchar   │   varchar   │  int64   │  timestamp   │ double │   varchar   │ varchar │ double  │
└─────────┴────────────┴─────────────┴──────────┴──────────────┴────────┴─────────────┴─────────┴─────────┘
                                                  0 rows                                                 

In [6]:
con.sql("""WITH COUNTS AS (
            SELECT
                (SELECT COUNT(*) FROM clean_sales) AS total_clean_rows,
                (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM clean_sales) AS t) AS unique_rows
)
       SELECT total_clean_rows,
               unique_rows,
               total_clean_rows - unique_rows AS duplicate_rows
       FROM COUNTS""").show()

┌──────────────────┬─────────────┬────────────────┐
│ total_clean_rows │ unique_rows │ duplicate_rows │
│      int64       │    int64    │     int64      │
├──────────────────┼─────────────┼────────────────┤
│          1003355 │     1003355 │              0 │
└──────────────────┴─────────────┴────────────────┘



In [7]:
con.sql("DESCRIBE clean_sales").show()

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ invoice      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ stock_code   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ description  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ quantity     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ invoice_date │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ price        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ country      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ revenue      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [8]:
con.sql("""SELECT customer_id
        FROM clean_sales
        LIMIT 5
        """)

┌─────────────┐
│ customer_id │
│   varchar   │
├─────────────┤
│ 14532       │
│ 17332       │
│ 14161       │
│ 13211       │
│ 16686       │
└─────────────┘

In [9]:
con.sql(""" SELECT COUNT (*) AS total_clean_rows,
        COUNT(*) - COUNT(customer_id) AS null_customer_rows,
        ROUND(100.0 * (COUNT(*) - COUNT(customer_id)) / COUNT(*), 1) AS pct_null
        FROM clean_sales""").show()

┌──────────────────┬────────────────────┬──────────┐
│ total_clean_rows │ null_customer_rows │ pct_null │
│      int64       │       int64        │  double  │
├──────────────────┼────────────────────┼──────────┤
│          1003355 │             226761 │     22.6 │
└──────────────────┴────────────────────┴──────────┘



## Summary

The clean_sales table has 1,003,355 rows. 6.0% (64,016) of raw_transactions total rows were removed. These rows were removed because they were not valid product sales. They were either cancellations, accounting adjustments, manual entries, vouchers, discounts, postage, tests, and duplicate lines with 34,335 rows (3.22%). 

All verification checks passed (0 invalid rows, 0 duplicates).

Rows with no customer ID (226,761, 22.6%) were kept for revenue analysis.

Added after analysis (04): standardised stock codes to uppercase, and removed 2 large orders that were cancelled within minutes (about £245k of sales that never happened).
